# Thai Voice Translator — Colab Server  🇹🇭
รัน backend ทั้งหมดบน Colab GPU T4 ฟรี + ngrok tunnel

### วิธีใช้:
1. เปิด [Colab](https://colab.research.google.com/) → File → Open notebook → GitHub → ใส่ `ponsatornsam/thai-voice-translator`
2. หรือ: File → Upload → เลือกไฟล์นี้
3. **Runtime → Change runtime type → T4 GPU**
4. ใส่ [ngrok authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) ใน Cell 6 (สมัครฟรี)
5. รันทีละ cell จากบนลงล่าง
6. เอา URL ที่ได้ไปใส่ใน Android app

> ⚠️ Colab session หมดอายุหลัง ~6-12 ชม. — ต้องรันใหม่ทุกครั้งที่หลุด

### Cell 1: ติดตั้ง dependencies (~2 นาที)

In [ ]:
!pip install -q fastapi uvicorn faster-whisper piper-tts httpx python-jose[cryptography] numpy soundfile pyngrok websockets
print("✅ Dependencies installed")

In [ ]:
# ติดตั้ง ffmpeg — ใช้สำหรับ Google TTS fallback
!apt install -y -qq ffmpeg 2>&1 | tail -1
print("✅ ffmpeg installed")

### Cell 2: ตั้งค่า API keys

In [ ]:
import os

os.environ["BACKEND_API_KEY"] = "your-backend-api-key-here"
os.environ["TRANSLATE_API_KEY"] = "your-gemini-api-key-here"
os.environ["TRANSLATE_API_BASE"] = "https://generativelanguage.googleapis.com/v1beta/openai"
os.environ["TRANSLATE_MODEL"] = "gemini-2.0-flash"
os.environ["PIPER_MODEL_DIR"] = "/content/models"
os.environ["PIPER_MODEL_NAME"] = "th_TH"

print("🔑 Keys configured")

### Cell 3: ดาวน์โหลด Piper Thai voice model (~1 นาที)

In [ ]:
import os
os.makedirs("/content/models", exist_ok=True)

# Piper binary
!wget -q https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_linux_x86_64.tar.gz -O /tmp/piper.tar.gz
!tar -xzf /tmp/piper.tar.gz -C /tmp/
!cp /tmp/piper/piper /usr/local/bin/piper
!chmod +x /usr/local/bin/piper
!rm -rf /tmp/piper /tmp/piper.tar.gz

# Thai voice model (medium quality)
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/th/th_TH/vits-th/th_TH-medium.onnx -O /content/models/th_TH.onnx
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/th/th_TH/vits-th/th_TH-medium.onnx.json -O /content/models/th_TH.json

!ls -lh /content/models/
!echo "Piper binary:" && /usr/local/bin/piper --help 2>&1 | head -1
print("\n✅ Piper model + binary ready")

### Cell 4: โหลด Whisper model บน GPU (~30 วิ)

In [ ]:
from faster_whisper import WhisperModel
import torch

if torch.cuda.is_available():
    print(f"🟢 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device, compute = "cuda", "int8"
else:
    print("🔴 No GPU! Runtime → Change runtime type → T4 GPU")
    device, compute = "cpu", "int8"

model = WhisperModel("base", device=device, compute_type=compute)
print(f"✅ Whisper model loaded on {device.upper()}")

### Cell 5: เริ่ม FastAPI server

In [ ]:
import sys, threading, time
import uvicorn

# Clone backend code
!rm -rf /content/repo /content/*.py
!git clone -q https://github.com/ponsatornsam/thai-voice-translator.git /content/repo
!cp /content/repo/backend/*.py /content/

sys.path.insert(0, "/content")

# Patch Piper model path for Colab
import tts_service
tts_service.MODEL_DIR = "/content/models"
tts_service.MODEL_NAME = "th_TH"
tts_service.MODEL_PATH = "/content/models/th_TH.onnx"
tts_service.MODEL_CONFIG = "/content/models/th_TH.json"
tts_service._model_available = True

# Patch: inject preloaded Whisper model
import whisper_service
whisper_service._model = model

def run_server():
    uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
print("✅ FastAPI server running on port 8000")

### Cell 6: สร้าง ngrok tunnel → ได้ URL สาธารณะ
**ต้องใส่ ngrok authtoken ก่อน** — ขอฟรีที่ https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok

# ⚠️ ใส่ authtoken ของคุณตรงนี้ (สมัครฟรีที่ ngrok.com)
NGROK_AUTHTOKEN = "your-ngrok-authtoken-here"

ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url
ws_url = public_url.replace("https://", "wss://").replace("http://", "ws://")

api_key = os.environ["BACKEND_API_KEY"]

print("=" * 65)
print("🔗  Android WebSocket URL (เอาไปใส่ใน app):")
print(f"    {ws_url}/ws/audio?api_key={api_key}")
print("=" * 65)
print(f"💚 Health check:      {public_url}/health")
print(f"📊 Rate limits:       {public_url}/admin/rate-limits?api_key={api_key}")
print(f"🔍 Pipeline status:   {public_url}/admin/pipeline-status?api_key={api_key}")
print("=" * 65)
print("⚠️  Session หมดอายุใน 6-12 ชม. — ต้องรันใหม่ทุกครั้ง")
print("⚠️  URL เปลี่ยนทุกครั้งที่รัน — ต้องอัปเดตใน Android app ด้วย")

### 🎉 เสร็จ! เซิร์ฟเวอร์กำลังรัน
- Colab cell 6 จะค้างอยู่ — **อย่าปิด tab นี้**
- เปิด Android Studio → เอา URL จากด้านบนไปใส่ใน `wsClient.connect()`
- Build APK → ทดสอบพูด → ได้ยินเสียงไทย!

**คราวหน้ามาใหม่:** เปิด notebook นี้ → รัน cell 1→6 ใหม่ (URL จะเปลี่ยน)